In [1]:
from Import_loader import *
import torch


Torch device: cpu

Available variants: ['scalar_rgb', 'scalar_spectral', 'llvm_ad_rgb', 'llvm_ad_spectral', 'llvm_mono']

Chosen variant: llvm_mono

Switched off Tx:  2
Switched off Rx:  10


In [2]:
# Experiment
utils.load_config("sphere_pos.json")
config["ref_position"] = 1

config["device"] = "cpu" # TODO: disable this again
config["use_sms"] = True # TODO: disable this again
# config["use_sms"] = False # TODO: disable this again

In [3]:
# Setup
params = setup_utils.setup_experiment(config)
scene = params['scene']
ref = params['ref_msr']
update_fn = params['update_fn']
gt = setup_utils.get_gt()

print(gt)

Loaded Reference: calibrated64/sphere_pos_1_foam_64f_330_mm
-0.10775


In [4]:
hyper_params = {
    'nsamples': 1,
    'sigma': 0.0125,
    'initial_translation': [0.0],
    'gt_translation': [gt],
    'learning_rate': 2e-2,
    'epochs': 50,
    'sigma_annealing': False,
    'anneal_const_first': 20,
    'anneal_const_last': 0,
    'anneal_sigma_min': 0.001,
}

ctx_args = {
    'scene': scene, 'init_pos': [0.0], 'gt_msr': ref,
    'update_fn': update_fn, 'loss_fn': losses.torch_fft_l2,
    'sampler': 'importance', 'antithetic': True, 'nsamples': hyper_params['nsamples'],
    'sigma': hyper_params['sigma'], 'device': config['device']
}

In [5]:
# Set up initial and gt translation
initial_translation = torch.tensor(hyper_params['initial_translation'], requires_grad=True, device=config['device'])
gt_translation = torch.tensor(hyper_params['gt_translation'], device=config['device'])

# Set up optimizer:
optim = torch.optim.Adam([initial_translation], lr=hyper_params['learning_rate'])

In [6]:
opt.test_sms_vs_multi(ctx_args)

# theta_optim = opt.run_optimization(
#     hparams=hyper_params,
#     optim=optim,
#     theta=initial_translation,
#     gt_theta=gt_translation,
#     ctx_args=ctx_args,
#     schedule_fn=opt.run_scheduler_step,
#     update_fn=update_fn,
# )

sms averages to 9.09s
multi averages to 5.21s


In [7]:
print(theta_optim)

NameError: name 'theta_optim' is not defined